In [1]:
import pandas as pd
import numpy as np

In [2]:
# starting_elo_df = pd.read_csv("starting_elo.csv")

In [3]:
# df = pd.read_csv("cleaned_data.csv", index_col=0)

In [4]:
# elo_container = starting_elo_df.iloc[0].to_dict()

In [5]:
# elo_container

In [6]:
# df["gameDateTimeEst"] = pd.to_datetime(df["gameDateTimeEst"])

In [7]:
def season_column(df):
    df["season"] = np.where(df["gameDateTimeEst"].dt.month >= 10, df["gameDateTimeEst"].dt.year + 1, df["gameDateTimeEst"].dt.year)
    return df

In [8]:
# df = season_column(df)
# df

In [9]:
def calculate_new_elo(old_elo, m, s, e, k=20):
    new_elo = old_elo + k * m * (s - e)

    return new_elo

In [10]:
def margin_of_victory(mov, elo_diff):
    m = ((mov + 3)**.8) / (7.5 + .006 * (elo_diff))
    
    return m

In [11]:
def off_season_regression(elo_container):
    for (team_name, current_elo) in elo_container.items():
        regressed_elo = (current_elo * .75) + (1505 * 0.25)

        elo_container[team_name] = regressed_elo
    return elo_container


In [12]:
def elo_calculation(df, elo_container):
    df = df.sort_values(by=["gameDateTimeEst", "gameId", "home"])
    pre_game_elos = []

    current_season = df["season"].iloc[0]
    for game_id, team_df in df.groupby("gameId", sort=False):

        game_season = team_df["season"].iloc[0]
        if current_season != game_season:
            off_season_regression(elo_container)
            current_season = game_season


        if len(team_df) != 2:
            continue


        home_row = team_df[team_df["home"] == 1].iloc[0]
        away_row = team_df[team_df["home"] == 0].iloc[0]

        home_name = home_row["teamName"]
        away_name = away_row["teamName"]

        home_pre = elo_container.get(home_name, 1500)
        away_pre = elo_container.get(away_name, 1500)

        pre_game_elos.extend([away_pre, home_pre])

        d = (home_pre + 100) - away_pre
        expected_home = 1 / (1 + 10 ** (-d / 400))
        expected_away = 1 - expected_home

        if home_row["win"] == 1:
            mov = home_row["points"] - home_row["opponentScore"]
            elo_diff = home_pre - away_pre
            home_s, away_s = 1, 0
        else:
            mov = away_row["points"] - away_row["opponentScore"]
            elo_diff = away_pre - home_pre
            home_s, away_s = 0, 1


        m = margin_of_victory(mov, elo_diff)

        home_post = calculate_new_elo(home_pre, m, home_s, expected_home)
        away_post = calculate_new_elo(away_pre, m, away_s, expected_away)


        elo_container[home_name] = home_post
        elo_container[away_name] = away_post

    df["pre_game_elo"] = pre_game_elos
    return df



In [13]:
# df = elo_calculation(df, elo_container)

In [14]:
# df.head()

In [15]:
# df.to_csv("data_with_elo.csv")

In [16]:
# idx = df.groupby("season")["pre_game_elo"].idxmax()
# result_df = df.loc[idx, ["teamName", "pre_game_elo", "season"]]
# result_df.sort_values(by=["season"], ascending=[False])